# [실습] 군집 — 고객을 묶고 의심하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 6 실습. 고객을 몇 개 그룹으로 나눠야 할까?

**필수 머신러닝 · Chapter 6**

| Part | 주제 |
| --- | --- |
| Part 1 | 고객을 숫자로: 정제와 RFM |
| Part 2 | K-means로 묶기 |
| Part 3 | 묶고 의심하기 |
| Part 4 | CRM팀에 보고하기 |

## 분석 의뢰서

> **의뢰 부서**: 온라인 도매몰 CRM팀
> **목적**: 고객 4,000여 명을 몇 개 그룹으로 나눠 그룹별로 다른 관리 정책(할인, 리마인드, 전담 상담)을 적용하고 싶다.
> **제약**: 그룹은 운영팀이 기억하고 관리할 수 있을 만큼 적어야 한다(5개 이내). 그룹이 바뀌면 정책도 바꿔야 하므로 자주 바뀌면 곤란하다.
> **질문**: 고객을 몇 개 그룹으로 나눠야 하고, 각 그룹은 어떤 고객이며, 그 그룹은 믿을 만한가?

## 이 실습의 질문

> **CRM팀에 줄 고객 그룹은 몇 개이고, 각 그룹의 기준은 무엇인가? 그 그룹이 "진짜"라고 CRM팀을 설득하는 근거는 무엇이고, 어디까지가 분석가의 선택인가?**

Chapter 1~5와 달리 이번에는 맞힐 정답이 없습니다. 그래서 "모델의 점수" 대신 **선택을 바꾸면 결과가 얼마나 바뀌는가**와 **군집에 쓰지 않은 데이터로 확인하기**에 시간을 씁니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

UCI Online Retail II 데이터를 CSV로 바꾼 `online_retail_II.csv`(약 97MB, 1,067,371행)를 이 노트북과 같은 폴더에 두세요. 파일이 크므로 Colab에서는 Google Drive에 올려 두고 연결해서 읽는 것을 권합니다.

In [46]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

---
# Part 1. 고객을 숫자로: 정제와 RFM

## 1-1. 데이터 불러오기

**[제공]** 송장 번호와 상품 코드는 글자로, 주문 일시는 날짜로 읽습니다. 행이 100만 개가 넘어 읽는 데 수십 초가 걸릴 수 있습니다.

In [47]:
# [제공] 데이터 불러오기
raw = pd.read_csv("/Users/limsunghun/python/DI/필수 머신러닝/1. 머신러닝이란 무엇인가/online_retail_II_all.csv", dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
print(raw.shape)
raw.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1-2. 정제: 실제로 산 기록만 남기기

**[직접 작성]** 다음 네 가지의 비율을 각각 출력한 뒤, 네 가지를 모두 뺀 데이터를 `sales`에 저장하세요. `sales`의 행 수와 고객 수(`CustomerID`의 고유값 수)도 출력합니다.

| 정제 항목 | 조건 |
| --- | --- |
| 고객 ID 없음 | `CustomerID`가 빈 값 |
| 취소 송장 | `Invoice`가 `C`로 시작 |
| 수량 0 이하 | `Quantity <= 0` |
| 가격 0 이하 | `Price <= 0` |

그다음 `sales`에 구매 금액 열 `amount`(= `Quantity` × `Price`)를 만드세요.

<details><summary>힌트</summary>

취소 송장은 `raw["Invoice"].str.startswith("C")`, 빈 값은 `raw["CustomerID"].isna()`. 비율은 조건에 `.mean()`을 붙입니다.
</details>

In [48]:
# 여기에 코드를 작성하세요

# 정제 조건 (raw 기준)
no_id    = raw["Customer ID"].isna()
canceled = raw["Invoice"].astype(str).str.startswith("C")
bad_qty  = raw["Quantity"] <= 0
bad_prc  = raw["Price"] <= 0

# 각 항목의 비율 출력
for name, cond in [("고객 ID 없음", no_id), ("취소 송장", canceled),
                   ("수량 0 이하", bad_qty), ("가격 0 이하", bad_prc)]:
    print(f"{name}: {cond.mean():.2%}")

# 네 조건을 모두 뺀 데이터
sales = raw[~(no_id | canceled | bad_qty | bad_prc)].copy()
print("행 수:", len(sales))
print("고객 수:", sales["Customer ID"].nunique())

# 구매 금액
sales["amount"] = sales["Quantity"] * sales["Price"]



고객 ID 없음: 22.77%
취소 송장: 1.83%
수량 0 이하: 2.15%
가격 0 이하: 0.58%
행 수: 805549
고객 수: 5878


> **[판단]** 반품을 빼지 않고 구매액에서 빼는 방식으로 정제했다면 군집 결과에 어떤 차이가 생길 수 있을까요? 이 정제 기준을 보고서에 적어야 하는 이유는 무엇인가요?
>
> 나의 답:

취소 송장은 전체 행의 1.83%에 불과해서 영향이 작아 보일 수 있다. 하지만 군집은 행이 아니라 고객 단위로 만들어지기 때문에, 반품이 일부 고객에게 몰려 있다면 그 고객들의 RFM값은 크게 달라진다.
지금 처럼 취소 행만 지우면 원래 구매 기록은 남아 있어서 반품이 많은 고객의 구매액이 실제보다 부풀려진다.

## 1-3. 기간 나누기와 RFM 계산

| 구간 | 기간 | 용도 |
| --- | --- | --- |
| 군집용 | 2009-12-01 ~ 2010-11-30 | 고객을 묶는다 |
| 검증용 | 2010-12-01 ~ 2011-11-30 | 묶은 그룹이 다음 해에 실제로 다르게 행동했는지 확인한다 |

**[직접 작성]** `sales`를 두 기간으로 나눠 `period1`, `period2`에 저장하세요. 그다음 **군집용 기간(`period1`)으로만** 고객마다 RFM을 계산해 `rfm`(고객 ID가 인덱스, 열 `R`, `F`, `M`)에 저장하고, 고객 수를 출력하세요. 기준일은 2010-12-01입니다.

| 지표 | 계산 |
| --- | --- |
| `R` | 기준일 − 마지막 구매일 (일) |
| `F` | 송장 수 (`Invoice`의 고유값 수) |
| `M` | 구매 금액의 합 (`amount`의 합) |

<details><summary>힌트</summary>

```python
ref = pd.Timestamp("2010-12-01")
rfm = period1.groupby("CustomerID").agg(
    R=("InvoiceDate", lambda s: (ref - s.max()).days),
    F=("Invoice", "nunique"),
    M=("amount", "sum"),
)
```
</details>

In [49]:
# 여기에 코드를 작성하세요
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])

period1 = sales[(sales["InvoiceDate"] >= "2009-12-01") & (sales["InvoiceDate"] < "2010-12-01")]
period2 = sales[(sales["InvoiceDate"] >= "2010-12-01") & (sales["InvoiceDate"] < "2011-12-01")]

ref = pd.Timestamp("2010-12-01")
rfm = period1.groupby("Customer ID").agg(
    R=("InvoiceDate", lambda s: (ref - s.max()).days),
    F=("Invoice", "nunique"),
    M=("amount", "sum"),
)
print("고객 수:", len(rfm))

고객 수: 4266


**[직접 작성]** `rfm`의 분포를 확인하세요. 최솟값, 중앙값, 평균, 상위 10%, 상위 1%, 최댓값, 표준편차가 보이도록 출력합니다.

<details><summary>힌트</summary>

`rfm.describe(percentiles=[0.5, 0.9, 0.99]).round(1)`
</details>

In [50]:
# 여기에 코드를 작성하세요

rfm.describe(percentiles=[0.5, 0.9, 0.99]).round(1)


,R,F,M
count,4266.0,4266.0,4266.0
mean,87.2,4.3,1997.2
std,95.1,7.8,8601.8
min,0.0,1.0,3.0
50%,47.0,2.0,689.1
90%,247.0,9.0,3683.4
99%,358.0,29.4,19518.9
max,364.0,197.0,321329.7


> **[판단]** F와 M의 평균과 중앙값은 얼마나 다른가요? 이 차이는 분포가 어떤 모양이라는 뜻인가요? 거리로 고객을 묶을 때 이 모양이 어떤 문제를 만들 것 같은가요?
>
> 나의 답:

F와 M의 평균 차이는 -1993정도가 차이가 나고 중앙값은 687정도 차이가 난다. 오른쪽으로 꼬리가 낀 모양의 그래프이다.

대다수 고객이 구분 되지 않는다. 중앙값 근처 고객들은 서로의 거리 차이가 극단값과의 거리에 비해 너무 작아서 하나의 큰 군집으로 뭉친다. 결과적으로 소수의 초대형 고객 대 나머지 전부 처럼 쓸모없는 구분이 나오기 쉽다.

---
# Part 2. K-means로 묶기

## 2-1. 손으로 돌려 보기

교안 2-1의 숫자 6개(1, 2, 3, 10, 11, 12)를 $k = 2$로 나눠 봅니다.

**[직접 작성]** `KMeans`에 초기 중심을 1과 2로 직접 주고(`init=np.array([[1], [2]])`, `n_init=1`) 학습한 뒤, 최종 중심, 그룹, inertia를 출력하세요. 이어서 같은 6개 점에 대해 `silhouette_score`로 실루엣 점수를 계산하세요.

In [51]:
# 여기에 코드를 작성하세요

X = np.array([1, 2, 3, 10, 11, 12], dtype=float).reshape(-1, 1)

km = KMeans(n_clusters=2, init=np.array([[1], [2]]), n_init=1)
km.fit(X)

print("최종 중심:", km.cluster_centers_.ravel())
print("그룹:", km.labels_)
print("inertia:", km.inertia_)
print("실루엣 점수:", silhouette_score(X, km.labels_))

최종 중심: [ 2. 11.]
그룹: [0 0 0 1 1 1]
inertia: 4.0
실루엣 점수: 0.850462962962963


> **[판단]** 나쁜 초기값(1과 2)에서 시작했는데도 최종 결과는 눈으로 봐도 자연스러운 두 덩어리인가요? 교안 2-1의 손 계산(중심 2와 11, inertia 4.0)과 같은가요?
>
> 나의 답:

자연스러운 두 덩어리이다. 최종그룹이 [0, 0, 0, 1, 1,1]로 {1, 2, 3}과 {10, 11, 12}로 나눠졌다. 이는 숫자를 눈으로 봤을 때 3과 10 사이의 큰 가격을 기준으로 나눴다.
교안 2-1의 손계산과 같은 값이 나왔다.

## 2-2. 전처리가 군집을 정한다

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 같은 K-means($k = 4$)를 세 가지 전처리에 적용합니다.

> **[판단]** 예상: RFM 원본 그대로 K-means를 돌리면 네 그룹의 크기는 어떻게 나올까요? (예: 비슷한 크기 네 개 / 하나는 아주 크고 나머지는 작음)
>
> 나의 답:

하나는 크고 나머지는 작은 모양이 나올 것 같다.

**[제공]** K-means를 돌려 그룹 크기와 실루엣을 보여 주는 함수입니다. `n_init=10`(여러 번 시작해 가장 좋은 결과), `random_state=42`로 고정합니다.

In [52]:
# [제공] K-means를 돌리고 그룹 크기와 실루엣을 출력
def run_kmeans(X, k, seed=42, verbose=True):
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
    if verbose:
        print(f"k={k} | 그룹 크기 {np.bincount(km.labels_)} | inertia {km.inertia_:,.0f} | 실루엣 {silhouette_score(X, km.labels_):.3f}")
    return km

**[직접 작성]** 세 가지 전처리로 행렬을 만들고 각각 `run_kmeans(X, 4)`를 실행하세요.

1. `X_raw`: R, F, M 원본 그대로
2. `X_std`: 세 열을 `StandardScaler`로 표준화
3. `X_log`: F와 M에 `np.log1p`를 적용한 뒤(R은 그대로) 세 열을 표준화

<details><summary>힌트</summary>

```python
X_log = StandardScaler().fit_transform(np.column_stack([rfm["R"], np.log1p(rfm["F"]), np.log1p(rfm["M"])]))
```
</details>

In [53]:
# 여기에 코드를 작성하세요

from sklearn.preprocessing import StandardScaler

X_raw = rfm[["R", "F", "M"]].to_numpy()
X_std = StandardScaler().fit_transform(X_raw)
X_log = StandardScaler().fit_transform(
    np.column_stack([rfm["R"], np.log1p(rfm["F"]), np.log1p(rfm["M"])])
)

res_raw = run_kmeans(X_raw, 4)
res_std = run_kmeans(X_std, 4)
res_log = run_kmeans(X_log, 4)

k=4 | 그룹 크기 [4137    3   13  113] | inertia 34,806,475,315 | 실루엣 0.885
k=4 | 그룹 크기 [3160 1049   52    5] | inertia 3,436 | 실루엣 0.607
k=4 | 그룹 크기 [1290  613  912 1451] | inertia 3,304 | 실루엣 0.373


> **[판단]** 예상은 맞았나요? 실루엣이 가장 높은 전처리는 무엇이고, 그 결과를 CRM팀에 줄 수 있나요? 실루엣이 높은데도 쓸 수 없다면 그 이유는 무엇인가요?
>
> 나의 답:

내 예상이 맞았다.

원본 그대로 돌린 결과 그룹 크기가 [4137, 3, 13, 113]으로 전체 4266명 중 97%가 한 그룹에 몰리고 나머지 세그룹은 3명, 13명, 113명이었다.

실루엣이 가장 높은 전처리는 원본데이터이다. 하지만 CRM팀에 줄 수 없다. 고객을 나눈 결과가 아니다. 97%가 한 그룹이라 CRM팀이 받아도 거의 모든 고객에게 같은 전략을 써야하고, 또한 3명짜리 그룹에 별도 캠페인을 설계할 수도 없다. 또한 사실상 M하나로만 나뉜 결과이다. 앞에서 예상한 대로 원본에서 M의 단위가 R,F보다 훨씬 커서 거리를 혼자 결정한다. 그래서 최근에 왔는지, 자주오는지 반영되지 않았다.

## 2-3. $k$ 고르기

**[직접 작성]** 로그 + 표준화한 `X_log`로 $k$를 2부터 8까지 바꿔 가며 `run_kmeans`를 실행하세요. inertia가 줄어드는 양(직전 $k$ 대비)도 계산해 보세요.

In [54]:
# 여기에 코드를 작성하세요

inertias = {}
for k in range(2, 9):
    km = run_kmeans(X_log, k)
    inertias[k] = km.inertia_

prev = None
for k, inertia in inertias.items():
    drop = "" if prev is None else f" | 감소량 {prev - inertia:,.0f} ({(prev - inertia) / prev:.1%})"
    print(f"k={k} | inertia {inertia:,.0f}{drop}")
    prev = inertia

k=2 | 그룹 크기 [1852 2414] | inertia 6,904 | 실루엣 0.397
k=3 | 그룹 크기 [1962 1334  970] | inertia 4,352 | 실루엣 0.409
k=4 | 그룹 크기 [1290  613  912 1451] | inertia 3,304 | 실루엣 0.373
k=5 | 그룹 크기 [ 880 1243  262  959  922] | inertia 2,838 | 실루엣 0.340
k=6 | 그룹 크기 [ 418  904  926  641  247 1130] | inertia 2,430 | 실루엣 0.340
k=7 | 그룹 크기 [ 871  880  415  248  365 1095  392] | inertia 2,172 | 실루엣 0.333
k=8 | 그룹 크기 [888 421 527 687 357 131 394 861] | inertia 1,948 | 실루엣 0.310
k=2 | inertia 6,904
k=3 | inertia 4,352 | 감소량 2,552 (37.0%)
k=4 | inertia 3,304 | 감소량 1,048 (24.1%)
k=5 | inertia 2,838 | 감소량 466 (14.1%)
k=6 | inertia 2,430 | 감소량 408 (14.4%)
k=7 | inertia 2,172 | 감소량 258 (10.6%)
k=8 | inertia 1,948 | 감소량 223 (10.3%)


> **[판단]** 엘보(inertia가 크게 줄다 완만해지는 지점)와 실루엣이 허용하는 후보는 무엇인가요? 숫자만으로 최종 $k$를 정할 수 있나요? 의뢰서의 제약(5개 이내, 그룹별로 다른 정책)을 넣으면 무엇을 고르겠습니까?
>
> 나의 답:

실루엣이 허용하는 후보는 3,4이다. 숫자만 으로 최종 k를 정할 수 없다.
의뢰서의 제약을 넣으면 k=4를 고를 것 같다.

## 2-4. 프로필과 이름 붙이기

**[직접 작성]** $k = 4$로 최종 모델을 만들어 그룹 번호를 `rfm["cluster"]`에 저장하고, 그룹별 고객 수와 R, F, M의 **중앙값**을 표로 출력하세요.

In [61]:
# 여기에 코드를 작성하세요

km4 = run_kmeans(X_log, 4)
rfm["cluster"] = final.labels_

profile = rfm.groupby("cluster").agg(
    고객수=("R", "size"),
    R=("R", "median"),
    F=("F", "median"),
    M=("M", "median"),
)
print(profile)

k=4 | 그룹 크기 [1290  613  912 1451] | inertia 3,304 | 실루엣 0.373
          고객수      R     F         M
cluster                             
0        1290   43.0   1.0   328.860
1         613   12.0  10.0  4214.770
2         912  243.0   1.0   286.275
3        1451   34.0   4.0  1307.400


**[직접 작성]** 위 프로필 표를 보고 그룹마다 이름을 붙여 `names` 딕셔너리에 저장하세요. 그룹 번호는 실행할 때마다 다를 수 있으니, **반드시 위 표를 보고** 번호와 이름을 맞춥니다. 이 이름은 Part 3에서 표를 읽기 쉽게 하는 데 씁니다.

예: `names = {0: "최근 1회 구매", 1: "핵심 고객", 2: "휴면 1회 구매", 3: "성장 중"}`

In [56]:
names = {0: "신규 고객", 1: "VIP 고객", 2: "휴면 고객", 3: "고객"}

**[제공]** 이름을 붙인 그룹 열 `group`을 만듭니다. (위 셀에서 이미 만들었다면 다시 실행해도 같습니다.)

In [57]:
# [제공] 이름 붙인 그룹 열
rfm["group"] = rfm["cluster"].map(names)
rfm["group"].value_counts()

group
고객        1451
신규 고객     1290
휴면 고객      912
VIP 고객     613
Name: count, dtype: int64

> **[판단]** 이름은 어떤 숫자를 근거로 붙였나요? 같은 표를 보고 다른 분석가는 다른 이름을 붙일 수 있을까요? 이름을 보고서에 쓸 때 함께 적어야 할 것은 무엇인가요?
>
> 나의 답:

이름은 R,F,M을 보고 정했다. 같은 표를 보고 다른 분석가는 다른 이름을 붙일 수 있다. 이름은 숫자가 아니라 숫자를 해석한결과이기 때문이다. 보고서에 이름을 쓸 때는 이름의 근거가 된 그룹별 고객 수 와 R, F, M 중앙값을 함께 적어야 한다. 또 이름이 데이터로 확인된 사실과 해석을 구분해야한다. 읽는 사람이 이름만 보고 판단하지 않고 숫자를 보고 직접 검증할 수 있어얗 한다.


---
# Part 3. 묶고 의심하기

## 3-1. 선택을 바꾸면 그룹이 바뀌는가

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다.

> **[판단]** 예상: $k = 3$의 그룹들은 $k = 4$에서 "하나가 둘로 쪼개지는" 모양일까요, 아니면 더 복잡하게 섞일까요? 그리고 초기값만 바꿔 다시 돌리면 결과가 달라질까요?
>
> 나의 답:

하나가 둘로 쪼개지는 모양이기 보다는 더 복잡하게 섞일 것이다.

K-means는 K를 바꾸면 처음부터 다시 학습하고, 모든 중심이 함께 움직인다. 계층적으로 그룹을 쪼개는 방법이 아니라서, k=3이 그룹 경계가 k=4에서  그대로 유지된다는 보장은 없다.

**[직접 작성]** 세 가지를 확인하세요.

1. $k = 3$으로 돌린 결과를 `rfm["k3"]`에 저장하고, `pd.crosstab(rfm["k3"], rfm["group"])`으로 $k = 4$의 그룹과 겹쳐 보기 ($k = 3$ 그룹의 R, F, M 중앙값도 함께 출력)
2. $k = 4$를 `random_state` 0~4로 다섯 번 다시 돌려, 최종 모델(`km4`)과의 ARI(`adjusted_rand_score`)
3. 계층적 군집(`AgglomerativeClustering(n_clusters=4, linkage="ward")`)과 최종 모델의 ARI

<details><summary>힌트</summary>

`run_kmeans(X_log, 4, seed=s, verbose=False).labels_`로 다시 돌린 결과를 얻고, `adjusted_rand_score(km4.labels_, 그 결과)`로 비교합니다.
</details>

In [62]:
# 여기에 코드를 작성하세요

# 1. k=3과 k=4 겹쳐 보기
rfm["k3"] = run_kmeans(X_log, 3, verbose=False).labels_
print(pd.crosstab(rfm["k3"], rfm["group"]))
print(rfm.groupby("k3")[["R", "F", "M"]].median())

# 2. 시드를 바꿔 다시 돌리기
for s in range(5):
    labels = run_kmeans(X_log, 4, seed=s, verbose=False).labels_
    print(f"seed={s} | ARI {adjusted_rand_score(km4.labels_, labels):.3f}")

# 3. 계층적 군집과 비교
ward = AgglomerativeClustering(n_clusters=4, linkage="ward").fit(X_log)
print(f"ward | ARI {adjusted_rand_score(km4.labels_, ward.labels_):.3f}")

group  VIP 고객   고객  신규 고객  휴면 고객
k3                              
0           0  716   1242      4
1         613  721      0      0
2           0   14     48    908
        R    F        M
k3                     
0    40.0  2.0   510.10
1    19.0  7.0  2486.95
2   235.0  1.0   280.10
seed=0 | ARI 1.000
seed=1 | ARI 0.997
seed=2 | ARI 1.000
seed=3 | ARI 0.998
seed=4 | ARI 0.999
ward | ARI 0.810


> **[판단]** 예상은 맞았나요? $k = 3$의 한 그룹이 $k = 4$에서 어떻게 나뉘고 합쳐지는지 숫자로 설명하세요. 초기값에 대한 ARI와 계층적 군집과의 ARI를 보고, 이 군집이 "안정적"인지와 "유일한 정답"인지를 구분해 말해 보세요.
>
> 나의 답:

예상이 맞았다. k=3에서 k=4로 깔끔하게 둘로 조개진 것이 아니라, 그룹이 k=3의 여러그룹에서 떼어 와 새로 만들어진다.

유일한 정답은 아니다. 방법을 Ward 계층적 군집으로 바꾸면 ARI가 0.81로 내려간다. 큰 구조는 비슷하지만 일부 고객은 다른 그룹에 묶였다는 뜻이다. 또 위의 교차표처럼 k를 3에서 4로 vip고객 그룹의 경계가 크게 바뀐다
 


## 3-2. 다음 해에 정말 달랐는가

정답이 없으니 "군집이 맞는가"는 물을 수 없습니다. 대신 "**군집이 앞으로의 행동을 가르는가**"를 봅니다. 검증용 기간(`period2`)은 군집에 한 번도 쓰지 않았습니다.

**[직접 작성]** 고객마다 검증용 기간의 구매 금액 합을 `rfm["next_amount"]`(구매가 없으면 0), 재구매 여부를 `rfm["repurchase"]`(구매 금액 > 0)에 저장하고, 그룹별로 고객 수, 다음 해 재구매율, 구매액 평균, 구매액 중앙값을 출력하세요. 전체 재구매율도 출력합니다.

<details><summary>힌트</summary>

`period2.groupby("CustomerID")["amount"].sum()`을 `.reindex(rfm.index).fillna(0)`으로 군집 대상 고객에 맞춥니다.
</details>

In [ ]:
# 여기에 코드를 작성하세요

> **[판단]** 다음 해 재구매율은 그룹마다 얼마나 다른가요? 특히 "휴면 1회 구매"와 "최근 1회 구매"는 F와 M이 거의 같은데 재구매율은 어떤가요? 이 결과는 $k = 4$를 고른 선택에 대해 무엇을 말해 주나요?
>
> 나의 답:

> **[판단]** 이 검증 결과가 말하지 **않는** 것을 두 가지 적어 보세요. (예: 그룹이 유일한가? 그룹이 원인인가? 정책이 효과가 있는가?)
>
> 나의 답:

## 3-3. 사람의 규칙과 비교하기

CRM 현장에서는 모델 없이 **RFM 점수**로 고객을 나눠 왔습니다. R, F, M을 고객 수가 같은 다섯 구간으로 나눠 1~5점을 주고(R은 최근일수록 5점), 점수 조합으로 세그먼트를 정합니다.

| 규칙 세그먼트 | 조건 |
| --- | --- |
| VIP | R 4~5점이고 F 4~5점 |
| 최근 신규 | R 4~5점 (F는 낮음) |
| 이탈 위험 우량 | R 1~2점이고 F 4~5점 |
| 휴면 | R 1~2점 (F는 낮음) |
| 일반 | 나머지 |

**[제공]** RFM 점수와 규칙 세그먼트를 만듭니다. F는 같은 값이 많아(절반이 2회 이하) 순위로 바꾼 뒤 다섯 구간으로 나눕니다.

In [ ]:
# [제공] RFM 점수와 규칙 세그먼트
r_score = pd.qcut(rfm["R"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
f_score = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
conditions = [
    (r_score >= 4) & (f_score >= 4),
    (r_score >= 4),
    (r_score <= 2) & (f_score >= 4),
    (r_score <= 2),
]
rfm["rule"] = np.select(conditions, ["VIP", "최근 신규", "이탈 위험 우량", "휴면"], default="일반")
rfm["rule"].value_counts()

**[직접 작성]** 두 가지를 출력하세요.

1. 규칙 세그먼트별 고객 수, R·F·M 중앙값, 다음 해 재구매율, 다음 해 구매액 평균
2. `pd.crosstab(rfm["rule"], rfm["group"])`으로 규칙 세그먼트와 K-means 그룹 겹쳐 보기

In [ ]:
# 여기에 코드를 작성하세요

> **[판단]** 규칙 세그먼트도 다음 해 행동을 가르나요? K-means의 "핵심 고객"과 규칙의 "VIP"는 어떻게 다른가요? K-means에는 없는데 규칙에는 있는 세그먼트는 무엇이고, 그 세그먼트는 CRM팀에 왜 중요할까요?
>
> 나의 답:

> **[판단]** 의뢰서의 제약 "그룹이 자주 바뀌면 곤란하다"를 생각하면 K-means와 규칙 중 어느 쪽이 유리한가요? 둘을 함께 쓴다면 어떻게 쓰겠습니까?
>
> 나의 답:

---
# Part 4. CRM팀에 보고하기

## 4-1. 쓸 수 있는 문장과 없는 문장

다음 문장들이 이 실습의 결과로 **쓸 수 있는 문장인지** 판단하고, 쓸 수 없다면 이유를 적으세요.

1. "고객을 R·F·M 기준으로 4개 그룹으로 나눴다. 각 그룹의 기준값은 표와 같다."
2. "데이터 분석 결과 우리 고객은 본래 4개 유형으로 나뉜다."
3. "네 그룹은 다음 해 재구매율이 35%에서 95%까지 다르다."
4. "초기값을 바꿔도 결과가 같았으므로(ARI 1.0에 가까움) 이 군집은 유일한 정답이다."
5. "이탈 위험 고객은 K-means 결과에 나타나지 않으므로 우리 고객 중에는 없다."

> **[판단]** 1~5번 각각: 쓸 수 있다 / 쓸 수 없다(이유)
>
> 나의 답:

## 4-2. CRM팀 보고 메모

분석 결과를 CRM팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **그룹 수와 각 그룹의 기준**: 이름과 함께 R·F·M 기준 숫자
2. **분석가의 선택**: 어떤 정보로, 어떻게 가공해, 왜 4개로 나눴는지 (다른 선택이 가능했다는 것도)
3. **믿을 만하다는 근거**: 군집에 쓰지 않은 다음 해 행동
4. **제안과 주의할 점**: 그룹별 정책을 어떻게 시험할지, 그룹 구분을 얼마나 자주 다시 볼지, 규칙과 함께 쓰는 방법

> **[판단]** CRM팀 보고 메모:
>
> 1. 그룹 수와 기준:
>
> 2. 분석가의 선택:
>
> 3. 믿을 만하다는 근거:
>
> 4. 제안과 주의할 점:
>
> 나의 답:

---
# 마무리

## 이 Chapter의 판단 포인트

> **군집은 발견이 아니라 선택이다. 전처리와 $k$를 바꾸면 그룹이 바뀐다. 그룹이 쓸모 있는지는 다음 기간의 행동으로 확인한다.**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "군집 결과를 보면 '전처리는 무엇이었고 $k$는 왜 그 값인가'를 먼저 묻는다."

> **[판단]** 나의 체크리스트 10번(군집 결과를 받았을 때):
>
> 나의 답:

> **[판단]** 나의 체크리스트 11번(정답이 없는 결과를 검증할 때):
>
> 나의 답:

## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 7에서는 차원 축소, 이상 탐지, 장바구니 분석으로 정답 없는 데이터에서 패턴을 찾고, 그 패턴이 내가 고른 기준의 결과인지 확인합니다.